# Graphs & Codes Playground
A browser-runnable tour of graph isomorphism tests and graph animation.

Open in Colab, then choose **Runtime → Run all**. The core examples are self-contained and do not access the private source repositories.

Inspired by [GI Lab](https://github.com/shrnasa011/testing_isomorphism) and [graph_animation](https://github.com/shrnasa011/graph_animation).

## 1. Can 1-WL distinguish these graphs?
The 4×4 rook graph and Shrikhande graph are non-isomorphic, but 1-WL gives them the same signature. Matching signatures do not prove isomorphism.

In [ ]:
from collections import Counter
import networkx as nx
import matplotlib.pyplot as plt


In [ ]:
def rook_graph(n=4):
    G = nx.Graph()
    G.add_nodes_from((i,j) for i in range(n) for j in range(n))
    for u in G:
        for v in G:
            if u < v and (u[0] == v[0] or u[1] == v[1]): G.add_edge(u,v)
    return G

def shrikhande_graph():
    steps={(1,0),(3,0),(0,1),(0,3),(1,1),(3,3)}
    G=nx.Graph(); G.add_nodes_from((i,j) for i in range(4) for j in range(4))
    for i,j in G:
        for a,b in steps: G.add_edge((i,j),((i+a)%4,(j+b)%4))
    return G

def wl1_signature(G, rounds=8):
    colors={v:G.degree(v) for v in G}; trace=[]
    for _ in range(rounds):
        trace.append(tuple(sorted(Counter(colors.values()).items())))
        keys={v:(colors[v],tuple(sorted(colors[u] for u in G.neighbors(v)))) for v in G}
        palette={key:i for i,key in enumerate(sorted(set(keys.values()),key=repr))}
        colors={v:palette[keys[v]] for v in G}
    trace.append(tuple(sorted(Counter(colors.values()).items())))
    return tuple(trace)

rook, shrikhande = rook_graph(), shrikhande_graph()
print('1-WL signatures match:', wl1_signature(rook)==wl1_signature(shrikhande))
print('Actually isomorphic:', nx.is_isomorphic(rook,shrikhande))

### Your turn
Try `nx.path_graph(6)` versus `nx.cycle_graph(6)`. Can you find graphs with the same degree sequence but different 1-WL signatures? Why can a matching signature not certify isomorphism?

In [ ]:
left, right = nx.path_graph(6), nx.cycle_graph(6)
print('1-WL signatures match:', wl1_signature(left)==wl1_signature(right))

## 2. Graph animation
First, explore a static, interactive graph drawing. Change the seed or selected vertex and rerun the cell.

In [ ]:
G=nx.erdos_renyi_graph(10,0.25,seed=7)
pos=nx.spring_layout(G,seed=7); chosen=max(G,key=G.degree)
fig,ax=plt.subplots(figsize=(7,4.5))
nx.draw_networkx_edges(G,pos,ax=ax,edge_color='#b9c7b5')
nx.draw_networkx_nodes(G,pos,ax=ax,node_color='#a6c39e',node_size=500)
nx.draw_networkx_nodes(G,pos,nodelist=[chosen],ax=ax,node_color='#e9a66b',node_size=700)
nx.draw_networkx_labels(G,pos,ax=ax)
ax.set_title(f'Highlighted vertex {chosen}, degree {G.degree[chosen]}'); ax.axis('off'); plt.show()

### Optional: render a Manim animation
This uses fresh demo code inspired by the graph-animation project. Manim installation and rendering may take a few minutes in Colab.

In [ ]:
%pip -q install manim

In [ ]:
from manim import *
import numpy as np

class GraphPlay(Scene):
    def construct(self):
        self.camera.background_color=WHITE
        G=nx.erdos_renyi_graph(7,0.35,seed=11); pos=nx.spring_layout(G,seed=11)
        pts={v:np.array([pos[v][0]*3,pos[v][1]*2,0]) for v in G}
        dots={v:Dot(pts[v],color=GREEN) for v in G}
        edges=VGroup(*[Line(pts[u],pts[v],color=GRAY) for u,v in G.edges])
        self.play(Create(edges),*[FadeIn(d) for d in dots.values()])
        v=max(G,key=G.degree)
        self.play(dots[v].animate.set_color(ORANGE).scale(1.5)); self.wait(1)

%%manim -ql -v WARNING GraphPlay

## Explore
1. Modify the refinement rule and observe which graph pairs it separates.
2. Change the graph generator and seed; highlight a different property.
3. Add your own graph pair and compare 1-WL against an exact isomorphism checker.

WL mismatches prove non-isomorphism; matches are inconclusive. The exact checker is intended for small examples only.

Source projects: [testing_isomorphism](https://github.com/shrnasa011/testing_isomorphism) · [graph_animation](https://github.com/shrnasa011/graph_animation). This notebook is self-contained and does not clone private repos.